In [1]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = os.path.join(r"c:\Users\priya\Downloads\ok\TinyBayes-ML", "data", "dataset", "cocoa")

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = ['anthracnose', 'cssvd', 'healthy']

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe from dataset directory
# ------------------------------------------------------------

data = []
for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    if os.path.exists(cls_dir):
        for img_file in os.listdir(cls_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({'Image_ID': img_file, 'class': cls_name})

df = pd.DataFrame(data)

print()
print("Total Images :", len(df))
print()
print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
NUM_CLASSES = len(label_encoder.classes_)

print()
print("Label Mapping")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split (Stratified 80 / 20)
# ------------------------------------------------------------

df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print()
print("="*60)
print("TRAIN SPLIT")
print("="*60)
print(df_train["class"].value_counts())

print()
print("="*60)
print("VALIDATION SPLIT")
print("="*60)
print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

df_train.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(SAVE_DIR, "validation_split.csv"), index=False)

print()
print("Saved")
print(os.path.join(SAVE_DIR, "train_split.csv"))
print(os.path.join(SAVE_DIR, "validation_split.csv"))


CLASSES
['anthracnose', 'cssvd', 'healthy']

Total Images : 5523

class
cssvd          2237
healthy        1720
anthracnose    1566

Label Mapping
0 -> anthracnose
1 -> cssvd
2 -> healthy

TRAIN SPLIT
class
cssvd          1789
healthy        1376
anthracnose    1253

VALIDATION SPLIT
class
cssvd          448
healthy        344
anthracnose    313

Saved
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\train_split.csv
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\validation_split.csv


In [3]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))


Overlap: 0


In [4]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)


Class names saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\class_names.json

Classes:
['anthracnose', 'cssvd', 'healthy']


In [5]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

if os.path.exists(TRAIN_FEATURE_PATH) and os.path.exists(VAL_FEATURE_PATH):

    print("="*60)
    print("FEATURE FILES FOUND")
    print("="*60)

    train_features_df = pd.read_csv(TRAIN_FEATURE_PATH)
    val_features_df = pd.read_csv(VAL_FEATURE_PATH)

    print(train_features_df.shape)
    print(val_features_df.shape)
    print()
    print(train_features_df.head())


FEATURE FILES FOUND
(4418, 578)
(1105, 578)

         Image_ID        class    feat_0    feat_1    feat_2
0   ID_Kf4dHB.jpg      healthy  0.592689  0.262962  0.541031
1  ID_GK394N.jpeg      healthy  0.105462  0.183077 -0.087441
2   ID_u21Vuy.jpg        cssvd  0.034122 -0.065241 -0.069107
3   ID_n2PA4u.jpg  anthracnose  0.107449  0.013330 -0.015691
4   ID_A397lM.jpg        cssvd -0.098958 -0.125566 -0.003058

Train features shape: (4418, 578)
Val features shape:   (1105, 578)


In [6]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------

feat_cols = [c for c in train_features_df.columns if c.startswith("feat_")]

X_train = train_features_df[feat_cols].values.astype(np.float32)
y_train = train_features_df["class"].values

X_val = val_features_df[feat_cols].values.astype(np.float32)
y_val = val_features_df["class"].values

# ------------------------------------------------------------
# One-hot encode targets
# ------------------------------------------------------------

y_train_df = pd.DataFrame({"c": y_train})
y_one_hot = pd.get_dummies(y_train_df["c"])

for cls in CLASS_NAMES:
    if cls not in y_one_hot.columns:
        y_one_hot[cls] = 0

y_one_hot = y_one_hot[CLASS_NAMES]

# ------------------------------------------------------------
# Jacobi regularized solve
# ------------------------------------------------------------

N = len(X_train)
a = 1.0 / N
b = 1.0 / N
k = 1.0

identity = np.eye(X_train.shape[1], dtype=np.float32)
regularization_lambda = 1.0

XtX = np.matmul(X_train.T, X_train)
XtX_reg = XtX + regularization_lambda * identity

jacobi_coefficients = {}

for cls in CLASS_NAMES:
    y_c = y_one_hot[cls].values
    eta = np.log((y_c + a) / (1.0 + k * b))
    Xty = np.matmul(X_train.T, eta)
    beta = np.linalg.solve(XtX_reg, Xty)
    jacobi_coefficients[cls] = beta

# ------------------------------------------------------------
# Validation inference
# ------------------------------------------------------------

preds_jacobi = []
for i in range(len(X_val)):
    x = X_val[i]
    scores = {cls: float(np.dot(x, jacobi_coefficients[cls])) for cls in CLASS_NAMES}
    preds_jacobi.append(max(scores, key=scores.get))

time_jacobi = time.time() - time_jacobi_start
acc_jacobi = accuracy_score(y_val, preds_jacobi)

print()
print("="*60)
print("JACOBI-DMR")
print("="*60)
print(f"Accuracy: {acc_jacobi:.4f}")
print(f"Time: {time_jacobi:.4f}s")
print()
print(classification_report(y_val, preds_jacobi, labels=CLASS_NAMES))
print()
print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_jacobi, labels=CLASS_NAMES))

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

JACOBI_JSON_PATH = os.path.join(SAVE_DIR, "jacobi_coefficients.json")

save_dict = {cls: [float(v) for v in jacobi_coefficients[cls]] for cls in CLASS_NAMES}

with open(JACOBI_JSON_PATH, "w") as f:
    json.dump(save_dict, f, indent=4)

print()
print("Jacobi coefficients saved to:")
print(JACOBI_JSON_PATH)



JACOBI-DMR
Accuracy: 0.8145
Time: 1.3377s

              precision    recall  f1-score   support

 anthracnose       0.84      0.82      0.83       313
       cssvd       0.83      0.81      0.82       448
     healthy       0.78      0.82      0.80       344

    accuracy                           0.81      1105
   macro avg       0.82      0.82      0.81      1105
weighted avg       0.82      0.81      0.81      1105


Confusion Matrix:

[[257  32  24]
 [ 29 362  57]
 [ 19  44 281]]

Jacobi coefficients saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\jacobi_coefficients.json


In [7]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_val)

time_rf = time.time() - time_rf_start

acc_rf = accuracy_score(y_val, preds_rf)

print()
print("="*60)
print("RANDOM FOREST")
print("="*60)

print(f"Accuracy: {acc_rf:.4f}")
print(f"Time: {time_rf:.4f}s")
print()

print(classification_report(y_val, preds_rf, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_rf, labels=CLASS_NAMES))



RANDOM FOREST
Accuracy: 0.7620
Time: 18.3286s

              precision    recall  f1-score   support

 anthracnose       0.82      0.66      0.73       313
       cssvd       0.76      0.82      0.79       448
     healthy       0.73      0.78      0.75       344

    accuracy                           0.76      1105
   macro avg       0.77      0.75      0.76      1105
weighted avg       0.77      0.76      0.76      1105


Confusion Matrix:

[[207  60  46]
 [ 28 367  53]
 [ 17  59 268]]


In [8]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)

preds_svm = svm_model.predict(X_val)

time_svm = time.time() - time_svm_start

acc_svm = accuracy_score(y_val, preds_svm)

print()
print("="*60)
print("SUPPORT VECTOR MACHINE")
print("="*60)

print(f"Accuracy: {acc_svm:.4f}")
print(f"Time: {time_svm:.4f}s")
print()

print(classification_report(y_val, preds_svm, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_svm, labels=CLASS_NAMES))



SUPPORT VECTOR MACHINE
Accuracy: 0.8353
Time: 15.3257s

              precision    recall  f1-score   support

 anthracnose       0.86      0.79      0.82       313
       cssvd       0.85      0.85      0.85       448
     healthy       0.80      0.86      0.83       344

    accuracy                           0.84      1105
   macro avg       0.84      0.83      0.83      1105
weighted avg       0.84      0.84      0.84      1105


Confusion Matrix:

[[248  32  33]
 [ 27 380  41]
 [ 14  35 295]]


In [9]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(X_train, y_train)

preds_ridge = ridge_model.predict(X_val)

time_ridge = time.time() - time_ridge_start

acc_ridge = accuracy_score(y_val, preds_ridge)

print()
print("="*60)
print("RIDGE CLASSIFIER")
print("="*60)

print(f"Accuracy: {acc_ridge:.4f}")
print(f"Time: {time_ridge:.4f}s")
print()

print(classification_report(y_val, preds_ridge, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_ridge, labels=CLASS_NAMES))



RIDGE CLASSIFIER
Accuracy: 0.8190
Time: 0.1990s

              precision    recall  f1-score   support

 anthracnose       0.85      0.81      0.83       313
       cssvd       0.83      0.81      0.82       448
     healthy       0.78      0.83      0.80       344

    accuracy                           0.82      1105
   macro avg       0.82      0.82      0.82      1105
weighted avg       0.82      0.82      0.82      1105


Confusion Matrix:

[[254  33  26]
 [ 26 365  57]
 [ 18  40 286]]


In [10]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lr_start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

preds_lr = lr_model.predict(X_val)

time_lr = time.time() - time_lr_start

acc_lr = accuracy_score(y_val, preds_lr)

print()
print("="*60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("="*60)

print(f"Accuracy: {acc_lr:.4f}")
print(f"Time: {time_lr:.4f}s")
print()

print(classification_report(y_val, preds_lr, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_lr, labels=CLASS_NAMES))



LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.8072
Time: 26.2628s

              precision    recall  f1-score   support

 anthracnose       0.82      0.81      0.81       313
       cssvd       0.82      0.81      0.82       448
     healthy       0.77      0.80      0.79       344

    accuracy                           0.81      1105
   macro avg       0.81      0.81      0.81      1105
weighted avg       0.81      0.81      0.81      1105


Confusion Matrix:

[[253  32  28]
 [ 31 365  52]
 [ 24  46 274]]


In [11]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)",
        "Jacobi-DMR",
        "Random Forest"
    ],
    "Accuracy": [
        acc_svm,
        acc_ridge,
        acc_lr,
        acc_jacobi,
        acc_rf
    ],
    "Execution Time (s)": [
        time_svm,
        time_ridge,
        time_lr,
        time_jacobi,
        time_rf
    ]
})

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison_df.to_string(index=False))

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()
print(f"Comparison saved to:\n{comparison_path}")



MODEL COMPARISON
                   Model  Accuracy  Execution Time (s)
  Support Vector Machine    0.8353             15.3257
        Ridge Classifier    0.8190              0.1990
              Jacobi-DMR    0.8145              1.3377
Logistic Regression (L1)    0.8072             26.2628
           Random Forest    0.7620             18.3286

Comparison saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\model_comparison.csv
